## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [1]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT



### Load Translator resources


In [2]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources()

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
imProving Agent for TRAPI 1.5 - no hits found in new metakg URL, trying old URL pattern
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0 - no hits found in new metakg URL, trying old URL pattern
CATRAX Pharmacogenomics KP - TRAPI 1.5.0 - no hits found in new metakg URL, trying old URL pattern
OpenPredict API - no hits found in new metakg URL, trying old URL pattern
mediKanren - no hits found in new metakg URL, trying old URL pattern
Knowledge Collaboratory API - no hits found in new metakg URL, trying old URL pattern
Sri-node-normalizer(Trapi v1.5.0) - no hits found in new metakg URL, trying old URL pattern
Shepherd SIPR - no hits found in new metakg URL, trying old URL pattern
Cqs(Trapi v1.5.0) - no hits found in new metakg URL, 

In [3]:
# select a list of APIs to use and a list of predicates to use
selected_APIlist = []

if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)
print(selected_metaKG.shape)

(31145, 5)


In [4]:

# sort APInames by name
APInames_sorted = dict(sorted(APInames.items()))
APInames_sorted


{'ARAX Translator Reasoner - TRAPI 1.6.0': 'https://arax.transltr.io/api/arax/v1.4/query/',
 'Answer-coalesce(Trapi v1.5.0)': 'https://answer-coalesce.transltr.io/query/',
 'Aragorn(Trapi v1.5.0)': 'https://aragorn.transltr.io/aragorn/query/',
 'Aragorn(Trapi v1.6.0)': 'https://aragorn.renci.org/aragorn/query/',
 'Automat-binding-db(Trapi v1.5.0)': 'https://automat.renci.org/binding-db/query/',
 'Automat-cam-kp(Trapi v1.5.0)': 'https://automat.transltr.io/cam-kp/query/',
 'Automat-ctd(Trapi v1.5.0)': 'https://automat.renci.org/ctd/query/',
 'Automat-drug-central(Trapi v1.5.0)': 'https://automat.renci.org/drugcentral/query/',
 'Automat-ehr-clinical-connections-kp(Trapi v1.5.0)': 'https://automat.renci.org/ehr-clinical-connections-kp/query/',
 'Automat-ehr-may-treat-kp(Trapi v1.5.0)': 'https://automat.renci.org/ehr-may-treat-kp/query/',
 'Automat-genome-alliance(Trapi v1.5.0)': 'https://automat.renci.org/genome-alliance/query/',
 'Automat-gtex(Trapi v1.5.0)': 'https://automat.renci.org/g

In [6]:
metaKG[metaKG['API']=='Retriever']

,API,Predicate,Subject,Object,URL
10624,Retriever,biolink:has_phenotype,biolink:Protein,biolink:PhenotypicFeature,https://retriever.ci.transltr.io/query/
10625,Retriever,biolink:has_phenotype,biolink:Gene,biolink:Disease,https://retriever.ci.transltr.io/query/
10626,Retriever,biolink:has_phenotype,biolink:Protein,biolink:Disease,https://retriever.ci.transltr.io/query/
10627,Retriever,biolink:has_substrate,biolink:SmallMolecule,biolink:Gene,https://retriever.ci.transltr.io/query/
10628,Retriever,biolink:has_substrate,biolink:Drug,biolink:Protein,https://retriever.ci.transltr.io/query/
...,...,...,...,...,...
14948,Retriever,biolink:affects,biolink:SmallMolecule,biolink:Gene,https://retriever.ci.transltr.io/query/
14949,Retriever,biolink:coexists_with,biolink:Gene,biolink:SmallMolecule,https://retriever.ci.transltr.io/query/
14950,Retriever,biolink:affects,biolink:Gene,biolink:SmallMolecule,https://retriever.ci.transltr.io/query/
14951,Retriever,biolink:interacts_with,biolink:Gene,biolink:SmallMolecule,https://retriever.ci.transltr.io/query/


In [37]:
All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))
# add 'Retriever' to API_withMetaKG
#API_withMetaKG.append('Retriever')
# generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

In [39]:
API_predicates['Retriever']

['biolink:located_in',
 'biolink:is_input_of',
 'biolink:ameliorates_condition',
 'biolink:part_of',
 'biolink:associated_with',
 'biolink:derives_from',
 'biolink:regulates',
 'biolink:acts_upstream_of_or_within',
 'biolink:causes',
 'biolink:involved_in',
 'biolink:occurs_together_in_literature_with',
 'biolink:interacts_with',
 'biolink:active_in',
 'biolink:studied_to_treat',
 'biolink:acts_upstream_of_or_within_positive_effect',
 'biolink:has_participant',
 'biolink:has_metabolite',
 'biolink:preventative_for_condition',
 'biolink:contraindicated_in',
 'biolink:expressed_in',
 'biolink:applied_to_treat',
 'biolink:directly_physically_interacts_with',
 'biolink:diagnoses',
 'biolink:treats_or_applied_or_studied_to_treat',
 'biolink:predisposes_to_condition',
 'biolink:acts_upstream_of_positive_effect',
 'biolink:gene_associated_with_condition',
 'biolink:enabled_by',
 'biolink:has_part',
 'biolink:has_substrate',
 'biolink:correlated_with',
 'biolink:genetic_association',
 'biolink

## Find the neighborhood of an entity from a subset of APIs 


In [40]:
# select a list of APIs to use and a list of predicates to use, if the list is empty, use all APIs and predicates
#selected_APIlist = ['Microbiome KP - TRAPI 1.5.0']
selected_APIlist = ['Retriever']
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]

print(select_APIs)
print(selected_metaKG.shape)


{'Retriever': 'https://retriever.ci.transltr.io/query/'}
(4329, 5)


In [ ]:
#name_resolver.lookup('BACE1', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')
#name_resolver.lookup('TP53', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('alzheimer disease')
name_resolver.lookup('acute myeloid leukemia', return_top_response=True, biolink_type='biolink:Disease',  limit=10) # sometimes the identifiers are not in the top 1, users need to check the other returned results


TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [44]:

#input_identifiers = 'MONDO:0004975'
input_identifiers = 'MONDO:0018874'

input_node_info = node_normalizer.get_normalized_nodes(input_identifiers)
input_node_info


TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing'], synonyms=None, curie_synonyms=None, attributes=None, taxa=None)


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [45]:
# to exclude BioThings Explorer (BTE) TRAPI: 
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT_neighborhood_finder.neighborhood_finder(input_identifiers,
                                                                                            node2_categories = ['biolink:Drug','biolink:SmallMolecule', 'biolink:ChemicalSubstance'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

MONDO:0018874


In [47]:
TCT_path_finder_result = TCT_neighborhood_finder.parse_results_for_neighborhood_finder(input_identifiers, result,
        start_node_categories='biolink:Disease', end_node_categories=None,
        get_node_info=True,
        scoring_method='infores')

In [48]:
TCT_path_finder_result

{'query_graph': {'nodes': {'on': {'categories': None,
    'constraints': [],
    'ids': [''],
    'is_set': False,
    'option_group_id': None,
    'set_id': None,
    'set_interpretation': 'BATCH'},
   'sn': {'categories': 'biolink:Disease',
    'constraints': [],
    'ids': ['MONDO:0018874'],
    'is_set': False,
    'option_group_id': None,
    'set_id': None,
    'set_interpretation': 'BATCH'}},
  'paths': {'p0': {'constraints': None,
    'object': 'on',
    'predicates': None,
    'subject': 'sn'}}},
 'knowledge_graph': {'nodes': {}, 'edges': {}},
 'results': [{'analyses': []}],
 'auxiliary_graphs': {}}

In [49]:
# write a result to a json file
import json
with open('TCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'new.json', 'w') as f:
    json.dump(TCT_path_finder_result, f)

In [44]:
# End of the example
